# 3. Managed Identities — secrets without the secrets

The biggest problem with the previous notebook? **That `client_secret`.** You have to:

- Store it somewhere (Key Vault, hopefully)
- Rotate it on schedule
- Keep it out of logs, git, env var dumps
- Give every environment (dev/staging/prod) a different one

**Managed identities** solve this. Azure creates a service principal for your resource and hands your code a token *on demand*, with no secret in your code or config.

> This notebook is mostly conceptual — managed identities only exist inside Azure (your app has to be running on App Service / Container App / VM / Function). The mock server simulates the moving pieces so you can *see* the shapes; when deployed to Azure the code becomes simpler, not more complex.

## System-assigned vs User-assigned

| | System-assigned | User-assigned |
|-|-----------------|----------------|
| Lifecycle | Born with the resource, dies with it | Standalone Azure resource |
| Sharing | 1 identity per resource | Attach to many resources |
| Use when | Single app, simple case | Shared identity across VMs / Container Apps, or you want to pre-grant permissions before deploying code |
| In Bicep | `identity: { type: 'SystemAssigned' }` | `identity: { type: 'UserAssigned', userAssignedIdentities: { '<id>': {} } }` |

**Rule of thumb**: default to user-assigned. You can grant the permissions and use them from CI before your app even exists, and you can attach the same identity to blue/green deployments.

## How your code gets a token inside Azure

Every Azure compute host exposes a metadata endpoint **IMDS** (Instance Metadata Service) at a fixed internal address:

- VMs / VMSS: `http://169.254.169.254/metadata/identity/oauth2/token`
- App Service / Functions / Container Apps: `$IDENTITY_ENDPOINT` env var

Your code calls this endpoint with a resource URI. The platform (not your code, not Entra) handles signing and returns a real Entra access token.

You almost never call IMDS directly — the Azure SDK does it for you via `ManagedIdentityCredential` or, more commonly, `DefaultAzureCredential`.

## The exact same code, three environments

The magic of `DefaultAzureCredential`: it tries multiple credential sources in order, returning the first that works. You ship one binary and it figures out which credential to use based on where it's running.

```python
from azure.identity import DefaultAzureCredential
cred = DefaultAzureCredential()
token = cred.get_token('api://api-b/.default').token
```

Credential chain (simplified):

1. **EnvironmentCredential** — `AZURE_CLIENT_ID` / `SECRET` / `TENANT_ID` env vars
2. **WorkloadIdentityCredential** — federated identity (AKS, GitHub Actions OIDC)
3. **ManagedIdentityCredential** — IMDS inside Azure compute
4. **AzureCliCredential** — whatever `az login` cached on your laptop
5. **VSCodeCredential** / **AzurePowerShellCredential** / ...

So:
- On your laptop after `az login` → uses AzureCliCredential
- In a Container App with a managed identity → uses ManagedIdentityCredential
- In CI/CD → EnvironmentCredential or WorkloadIdentityCredential

**Zero code changes between them.**

## Simulating managed identity locally

For this lab we can't spin up IMDS. Instead we use **EnvironmentCredential** (first in the chain) — set the env vars and the SDK behaves identically to a Service Principal login. In real Azure these vars aren't set, so it skips to ManagedIdentityCredential.

Let's prove it works with real `DefaultAzureCredential` code pointed at our mock Entra.

In [ ]:
import os, httpx, json, base64

# Feed the same values to EnvironmentCredential via env vars.
os.environ['AZURE_CLIENT_ID']      = 'daemon-client-id'
os.environ['AZURE_CLIENT_SECRET']  = 'daemon-secret-value'
os.environ['AZURE_TENANT_ID']      = 'contoso'
# Point the SDK's authority at our mock (only needed because it's not real Entra):
os.environ['AZURE_AUTHORITY_HOST'] = 'http://localhost:9000'

from azure.identity import EnvironmentCredential
cred = EnvironmentCredential()

# The SDK validates authority certs by default; our mock is HTTP so we'll
# get the token the manual way to keep the demo focused. In real Azure:
#     token = cred.get_token('api://api-b/.default').token

r = httpx.post('http://localhost:9000/contoso/oauth2/v2.0/token', data={
    'grant_type': 'client_credentials',
    'client_id': os.environ['AZURE_CLIENT_ID'],
    'client_secret': os.environ['AZURE_CLIENT_SECRET'],
    'scope': 'api://api-b/.default',
})
token = r.json()['access_token']
print('Got token via SP credentials (simulating managed identity output).')

r = httpx.get('http://localhost:8002/files', headers={'Authorization': f'Bearer {token}'})
print(json.dumps(r.json(), indent=2))

## What the deployment looks like in Bicep

```bicep
// 1. Create a user-assigned managed identity
resource uami 'Microsoft.ManagedIdentity/userAssignedIdentities@2023-01-31' = {
  name: 'my-worker-identity'
  location: location
}

// 2. Attach it to a Container App
resource app 'Microsoft.App/containerApps@2024-03-01' = {
  name: 'worker'
  location: location
  identity: {
    type: 'UserAssigned'
    userAssignedIdentities: { '${uami.id}': {} }
  }
  properties: { /* ... image, env, ... */ }
}

// 3. In the Entra app registration for api-b, grant this identity the
//    'Files.Read.All' app role. This is done via an azureADServicePrincipal
//    appRoleAssignment - typically by a post-deployment script or az CLI.
```

## Granting app roles to a managed identity (CLI)

```bash
# Get the object id of the managed identity
MI_PRINCIPAL_ID=$(az identity show -g my-rg -n my-worker-identity --query principalId -o tsv)

# Get the service principal + role id for api-b
API_B_SP=$(az ad sp list --filter "displayName eq 'api-b'" --query '[0].id' -o tsv)
ROLE_ID=$(az ad sp show --id $API_B_SP --query "appRoles[?value=='Files.Read.All'].id" -o tsv)

# Assign
az rest --method POST \
  --uri "https://graph.microsoft.com/v1.0/servicePrincipals/$MI_PRINCIPAL_ID/appRoleAssignments" \
  --body "{\"principalId\":\"$MI_PRINCIPAL_ID\",\"resourceId\":\"$API_B_SP\",\"appRoleId\":\"$ROLE_ID\"}"
```

## Debug checklist when it doesn't work in Azure

1. Is the identity actually attached? `az containerapp identity show -g <rg> -n <app>`
2. Does the identity have the role granted? See CLI snippet above.
3. Is your code asking for the right `/.default` scope?
4. Does `aud` in the issued token match what the API validates? Use the `/debug/decode/<token>` endpoint or [jwt.ms](https://jwt.ms).
5. Is the JWKS URI reachable from your app's network?

## Summary

- Managed identity = no secrets, Azure-managed SP + IMDS.
- Prefer **user-assigned** for anything non-trivial.
- `DefaultAzureCredential` = one code path for local + Azure.
- In the token, a managed identity looks identical to any client-credentials token (`roles` claim, no user).